# 05 - Spark optimization

В этой практике нужно не просто ускорить один `groupBy`, а провести полноценное мини-исследование производительности Spark pipeline.

Ожидаемый объем решения: примерно 180-260 строк Python/Spark-кода без учета markdown.


## Scenario

Используй наборы `orders.csv`, `order_items.csv`, `products.json`, при желании `events.json`.
Нужно построить pipeline, который:
- читает несколько источников;
- строит wide transformations;
- считает витрину продаж по дням, категориям и типам событий;
- сравнивает baseline и optimized pipeline;
- сравнивает как минимум две конфигурации Spark.


In [ ]:
import time
from typing import Dict, List

from pyspark.sql import SparkSession, DataFrame, functions as F

DATASET_BASE = "/workspace/dataset"
ORDERS_PATH = f"{DATASET_BASE}/orders.csv"
ORDER_ITEMS_PATH = f"{DATASET_BASE}/order_items.csv"
PRODUCTS_PATH = f"{DATASET_BASE}/products.json"
EVENTS_PATH = f"{DATASET_BASE}/events.json"
OUTPUT_BASE = "file:///workspace/output/lesson05"


## Step 1 - Spark configs

Подготовь минимум две конфигурации:
- baseline;
- tuned.

В tuned-конфиге обязательно поработай с `spark.sql.shuffle.partitions` и еще минимум двумя параметрами.


In [ ]:
BASELINE_CONFIG: Dict[str, str] = {
    "spark.sql.shuffle.partitions": "200",
    "spark.sql.adaptive.enabled": "false",
}

TUNED_CONFIG: Dict[str, str] = {
    "spark.sql.shuffle.partitions": "32",
    "spark.sql.adaptive.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.enabled": "true",
    "spark.sql.adaptive.skewJoin.enabled": "true",
    "spark.sql.autoBroadcastJoinThreshold": str(50 * 1024 * 1024),
    "spark.sql.files.maxPartitionBytes": str(32 * 1024 * 1024),
}


In [ ]:
def build_spark(app_name: str, config: Dict[str, str]) -> SparkSession:
    builder = SparkSession.builder.appName(app_name)
    for key, value in config.items():
        builder = builder.config(key, value)
    return builder.getOrCreate()


## Step 2 - Input readers and helper functions

Сделай helpers для чтения и нормализации входных наборов данных.


In [ ]:
def read_inputs(spark: SparkSession) -> Dict[str, DataFrame]:
    orders_df = (
        spark.read.option("header", True).option("inferSchema", True).csv(ORDERS_PATH)
    )
    order_items_df = (
        spark.read.option("header", True).option("inferSchema", True).csv(ORDER_ITEMS_PATH)
    )
    products_df = spark.read.json(PRODUCTS_PATH)
    events_df = spark.read.json(EVENTS_PATH)
    return {
        "orders": orders_df,
        "order_items": order_items_df,
        "products": products_df,
        "events": events_df,
    }


def print_dataset_overview(frames: Dict[str, DataFrame]) -> None:
    for name, df in frames.items():
        print(f"=== {name} ===")
        print("rows:", df.count())
        df.printSchema()
        df.show(3, truncate=False)


## Step 3 - Baseline pipeline

Построй заведомо тяжелый pipeline:
- несколько join;
- широкая агрегация;
- лишние колонки до join;
- возможно, повторное использование без cache.


In [ ]:
def build_baseline_pipeline(frames: Dict[str, DataFrame]) -> DataFrame:
    orders_df = frames["orders"]
    order_items_df = frames["order_items"]
    products_df = frames["products"]
    events_df = frames["events"]

    exploded_events = (
        events_df
        .withColumn("item", F.explode_outer("items"))
        .select(
            F.col("event_id"),
            F.col("ts").alias("event_ts"),
            F.col("type").alias("event_type"),
            F.col("user_id").alias("event_user_id"),
            F.col("item.product_id").alias("event_product_id"),
            F.col("item.qty").alias("event_qty"),
            F.col("item.price").alias("event_price"),
            F.col("device"),
        )
    )

    joined_df = (
        order_items_df
        .join(orders_df, on="order_id", how="inner")
        .join(products_df, on="product_id", how="left")
        .join(
            exploded_events,
            on=order_items_df.product_id == exploded_events.event_product_id,
            how="left",
        )
    )

    baseline_df = (
        joined_df
        .groupBy("dt", "category", "event_type", "device")
        .agg(
            F.count("*").alias("rows_cnt"),
            F.round(F.sum(F.col("qty") * F.col("price")), 2).alias("revenue"),
            F.round(F.avg("price"), 2).alias("avg_price"),
            F.countDistinct("order_id").alias("orders_cnt"),
            F.countDistinct("user_id").alias("users_cnt"),
        )
        .orderBy("dt", "category", "event_type")
    )
    return baseline_df


In [ ]:
def timed_action(label: str, df: DataFrame) -> Dict[str, float]:
    started = time.perf_counter()
    row_cnt = df.count()
    elapsed = time.perf_counter() - started
    print(f"[{label}] rows={row_cnt} elapsed={elapsed:.2f}s")
    return {"label": label, "rows": row_cnt, "elapsed_sec": round(elapsed, 2)}


In [ ]:
# TODO:
# 1. Подними Spark c BASELINE_CONFIG
# 2. Прочитай входные данные
# 3. Собери baseline_df = build_baseline_pipeline(...)
# 4. Вызови baseline_df.explain(mode="formatted")
# 5. Зафиксируй baseline metrics через timed_action(...)


## Step 4 - Code-level optimizations

Сделай минимум три улучшения:
- early projection/select;
- broadcast маленького справочника;
- cache или persist для повторно используемого слоя;
- управление partitioning перед тяжелой агрегацией.


In [ ]:
def build_optimized_pipeline(frames: Dict[str, DataFrame]) -> DataFrame:
    orders_df = frames["orders"].select("order_id", "dt", "user_id")
    order_items_df = frames["order_items"].select("order_id", "product_id", "qty", "price")
    products_df = frames["products"].select("product_id", "category")
    events_df = frames["events"]

    event_items_df = (
        events_df
        .filter(F.col("type").isin("cart", "purchase"))
        .withColumn("item", F.explode_outer("items"))
        .select(
            F.col("type").alias("event_type"),
            F.col("device"),
            F.col("item.product_id").alias("product_id"),
        )
        .dropna(subset=["product_id"])
    )

    fact_df = (
        order_items_df
        .join(orders_df, on="order_id", how="inner")
        .join(F.broadcast(products_df), on="product_id", how="left")
    ).cache()

    enriched_df = (
        fact_df
        .join(event_items_df.repartition("product_id"), on="product_id", how="left")
        .fillna({"event_type": "no_event", "device": "unknown"})
    )

    result_df = (
        enriched_df
        .groupBy("dt", "category", "event_type", "device")
        .agg(
            F.count("*").alias("rows_cnt"),
            F.round(F.sum(F.col("qty") * F.col("price")), 2).alias("revenue"),
            F.round(F.avg("price"), 2).alias("avg_price"),
            F.countDistinct("order_id").alias("orders_cnt"),
            F.countDistinct("user_id").alias("users_cnt"),
        )
        .orderBy("dt", "category", "event_type")
    )
    return result_df


In [ ]:
# TODO:
# 1. Собери optimized_df = build_optimized_pipeline(...)
# 2. Вызови optimized_df.explain(mode="formatted")
# 3. Зафиксируй метрики optimized version
# 4. Опиши, какие code-level изменения реально поменяли physical plan


## Step 5 - Config-level tuning

Сравни поведение pipeline при разных Spark config.


In [ ]:
def collect_runtime_report(config_name: str, config: Dict[str, str]) -> Dict[str, object]:
    spark = build_spark(f"lesson05_{config_name}", config)
    frames = read_inputs(spark)
    baseline_df = build_baseline_pipeline(frames)
    optimized_df = build_optimized_pipeline(frames)
    baseline_metrics = timed_action(f"{config_name}_baseline", baseline_df)
    optimized_metrics = timed_action(f"{config_name}_optimized", optimized_df)
    return {
        "config_name": config_name,
        "config": config,
        "baseline_metrics": baseline_metrics,
        "optimized_metrics": optimized_metrics,
    }


In [ ]:
def flatten_report(report: Dict[str, object]) -> Dict[str, object]:
    return {
        "config_name": report["config_name"],
        "shuffle_partitions": report["config"].get("spark.sql.shuffle.partitions"),
        "aqe_enabled": report["config"].get("spark.sql.adaptive.enabled"),
        "baseline_rows": report["baseline_metrics"]["rows"],
        "baseline_elapsed_sec": report["baseline_metrics"]["elapsed_sec"],
        "optimized_rows": report["optimized_metrics"]["rows"],
        "optimized_elapsed_sec": report["optimized_metrics"]["elapsed_sec"],
    }

def compare_reports(reports: List[Dict[str, object]]):
    # TODO: импортируй pandas и собери итоговую таблицу сравнения
    # TODO: добавь колонки improvement_sec и improvement_pct
    raise NotImplementedError


In [ ]:
# TODO:
# 1. Прогони collect_runtime_report("baseline_config", BASELINE_CONFIG)
# 2. Прогони collect_runtime_report("tuned_config", TUNED_CONFIG)
# 3. Сравни, как конфигурация влияет на baseline и optimized pipeline


## Step 6 - Benchmark table

Собери итоговую таблицу сравнения и короткий инженерный вывод.


In [ ]:
# TODO: собери список словарей с метриками и преврати его в pandas DataFrame
# TODO: сравни elapsed_sec, rows, config_name, explain notes
# TODO: запиши итоговую таблицу в parquet/csv при желании


In [ ]:
report_template = {
    "baseline_bottlenecks": ["TODO"],
    "code_level_findings": ["TODO"],
    "config_level_findings": ["TODO"],
    "recommended_defaults": {
        "spark.sql.shuffle.partitions": "TODO",
        "spark.sql.adaptive.enabled": "TODO",
        "spark.sql.autoBroadcastJoinThreshold": "TODO",
    },
}
report_template


In [ ]:
summary = {
    "main_shuffle_source": "TODO",
    "best_code_optimization": "TODO",
    "best_config_change": "TODO",
    "when_broadcast_is_safe": "TODO",
    "when_more_partitions_is_bad": "TODO",
}
summary
